# Notebook 3: Regenerate and save figures

This notebook does not refit the encoding models. It loads the files saved by Notebook 2, rebuilds the figures, and saves them to `/kaggle/working/figures`.


## 0. Setup and load


In [ ]:
!pip install nilearn --quiet

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting

RESULTS = Path("/kaggle/input/datasets/delhialli/podcast-ecog-encoding-results")
FIGURES = Path("/kaggle/working/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

with open(RESULTS / "results.json") as file:
    saved = json.load(file)

R = saved["whisper"]
Q = saved["qwen"]

per_electrode = np.load(RESULTS / "per_electrode.npz", allow_pickle=True)

with open(RESULTS / "run_meta.json") as file:
    metadata = json.load(file)

print("Whisper layers:", metadata["whisper_layers"])
print("Qwen layers:", metadata["qwen_layers"])
print("Figures will be saved to:", FIGURES)


## 1. Whisper analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(R["layer"], R["redundancy"], "o-")
axes[0].set(
    title="Redundancy with five interpretable features",
    xlabel="normalized layer",
    ylabel="shared R² / model R²",
)
axes[0].set_ylim(bottom=0)

axes[1].plot(R["layer"], R["unique"], "o-")
axes[1].set(
    title="Whisper-unique variance",
    xlabel="normalized layer",
    ylabel="mean unique R²",
)
axes[1].set_ylim(bottom=0)

axes[2].plot(R["layer"], R["r_model"], "o-", label="Whisper")
axes[2].plot(R["layer"], R["r_joint"], "s-", label="Whisper + interpretable")
axes[2].set(
    title="Whisper encoding accuracy",
    xlabel="normalized layer",
    ylabel="mean Pearson r",
)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "whisper_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Speech versus text

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for results, style, model_name in [
    (R, "o-", "Whisper"),
    (Q, "s--", "Qwen"),
]:
    axes[0].plot(results["layer"], results["mel"], style, label=f"{model_name}: mel")
    axes[0].plot(results["layer"], results["gbfb"], style, label=f"{model_name}: GBFB")

axes[0].set(
    title="Acoustic overlap after controls",
    xlabel="normalized layer",
    ylabel="mean shared R²",
)
axes[0].set_ylim(bottom=0)
axes[0].legend(fontsize=8)

for results, style, model_name in [
    (R, "o-", "Whisper"),
    (Q, "s--", "Qwen"),
]:
    axes[1].plot(results["layer"], results["phonetic"], style, label=f"{model_name}: phonetic")
    axes[1].plot(results["layer"], results["syntactic"], style, label=f"{model_name}: syntactic")

axes[1].set(
    title="Linguistic overlap after controls",
    xlabel="normalized layer",
    ylabel="mean shared R²",
)
axes[1].set_ylim(bottom=0)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "controlled_overlap.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Qwen accuracy and participant check

In [ ]:
col_subject = per_electrode["col_subject"]
subjects = sorted(np.unique(col_subject).tolist())

final_syntax_whisper = per_electrode["final_syntax_shared_whisper"]
final_syntax_qwen = per_electrode["final_syntax_shared_qwen"]

whisper_subject_syntax = np.array([
    final_syntax_whisper[col_subject == subject].mean()
    for subject in subjects
])
qwen_subject_syntax = np.array([
    final_syntax_qwen[col_subject == subject].mean()
    for subject in subjects
])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(Q["layer"], Q["r_model"], "o-", label="Qwen")
axes[0].plot(Q["layer"], Q["r_joint"], "s-", label="Qwen + interpretable")
axes[0].set(
    title="Qwen encoding accuracy",
    xlabel="normalized layer",
    ylabel="mean Pearson r",
)
axes[0].legend(fontsize=8)

for i, subject in enumerate(subjects):
    axes[1].plot(
        [0, 1],
        [whisper_subject_syntax[i], qwen_subject_syntax[i]],
        "o-",
    )

axes[1].set_xticks([0, 1], ["Whisper L32", "Qwen L28"])
axes[1].set(
    title="Final syntactic overlap by participant",
    ylabel="mean shared R²",
)
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(FIGURES / "qwen_and_participants.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Electrode maps

In [ ]:
xyz = per_electrode["xyz"]
has_coordinate = per_electrode["has_coordinate"]


def plot_map(values, title, filename, cmap="Reds", vmin=0, vmax=None, shown=None):
    if shown is None:
        shown = has_coordinate

    shown_values = values[shown]
    positions = xyz[shown]

    if vmax is None:
        positive = shown_values[shown_values > 0]
        if not len(positive):
            print(title, ": no positive values")
            return
        vmax = np.percentile(positive, 99)

    order = np.argsort(shown_values)

    display = plotting.plot_markers(
        shown_values[order],
        positions[order],
        node_cmap=cmap,
        node_vmin=vmin,
        node_vmax=vmax,
        node_size=20,
        title=title,
    )
    display.savefig(FIGURES / filename, dpi=300)
    plotting.show()

In [ ]:
unique_whisper = per_electrode["unique_whisper_beyond_interp"]
unique_qwen = per_electrode["unique_qwen_beyond_interp"]

positive = np.concatenate([
    unique_whisper[has_coordinate],
    unique_qwen[has_coordinate],
])
positive = positive[positive > 0]
shared_vmax = np.percentile(positive, 99)

plot_map(
    unique_whisper,
    "Whisper L32 unique variance beyond interpretable features",
    "whisper_unique_map.png",
    vmax=shared_vmax,
)

plot_map(
    unique_qwen,
    "Qwen L28 unique variance beyond interpretable features",
    "qwen_unique_map.png",
    vmax=shared_vmax,
)

### Preferred layers among the five sampled depths


In [ ]:
for model_name in ["whisper", "qwen"]:
    values = per_electrode[f"preferred_layer_{model_name}"]
    peak_r = per_electrode[f"peak_r_{model_name}"]
    shown = has_coordinate & (peak_r > 0.05)

    if shown.any():
        plot_map(
            values,
            f"Preferred {model_name.capitalize()} layer among five sampled depths",
            f"preferred_layer_{model_name}.png",
            cmap="viridis",
            vmin=0,
            vmax=1,
            shown=shown,
        )

## 5. Numerical summary

In [ ]:
print("WHISPER REPLICATION")
print(
    "  redundancy, first -> last:",
    round(R["redundancy"][0], 3),
    "->",
    round(R["redundancy"][-1], 3),
)
print(
    "  model-unique, first -> last:",
    round(R["unique"][0], 5),
    "->",
    round(R["unique"][-1], 5),
)
print(
    "  accuracy, first -> last:",
    round(R["r_model"][0], 3),
    "->",
    round(R["r_model"][-1], 3),
)

rows = []
for model_name, results in [("Whisper L32", R), ("Qwen L28", Q)]:
    for feature in ["mel", "gbfb", "phonetic", "syntactic"]:
        rows.append({
            "model": model_name,
            "feature": feature,
            "shared R²": results[feature][-1],
            "feature R² after controls": results[f"{feature}_denom"][-1],
        })

summary = pd.DataFrame(rows).round(6)
summary.to_csv(FIGURES / "final_layer_overlap.csv", index=False)

print("\nFINAL-LAYER CONTROLLED OVERLAP")
display(summary)

print("\nSaved files:")
for path in sorted(FIGURES.iterdir()):
    print(" ", path)